<a href="https://colab.research.google.com/github/prasertrak/Advanced-Data-Engineering-and-Applied-Analytics/blob/main/Lab2_sql_python_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — SQL + Python for Data Engineers
### Order Fulfillment Analytics Pipeline / Simulate ETL Pipeline


### Learning Objectives
- Connect Python with SQL databases
- Read data from database
- Insert records into tables
- Execute SQL queries
- Perform JOIN and Aggregation
- Simulate ETL pipeline


In [ ]:
import pandas as pd
import sqlite3

## 1. Create SQLite Database Connection

In [ ]:
conn = sqlite3.connect("lab2.db")

print("Database Connected")


Database Connected


## 2. Create Orders Table

In [ ]:
drop_orders_table = '''
DROP TABLE IF EXISTS orders;
'''

conn.execute(drop_orders_table)

create_orders_table = '''
CREATE TABLE orders (
    order_id          TEXT PRIMARY KEY,
    customer_id       TEXT NOT NULL,
    order_date        TEXT NOT NULL,
    product_category  TEXT NOT NULL,
    quantity          INTEGER NOT NULL CHECK (quantity > 0),
    unit_price        REAL NOT NULL CHECK (unit_price >= 0),
    delivery_days     INTEGER CHECK (delivery_days >= 0),
    status            TEXT NOT NULL
);
'''

conn.execute(create_orders_table)

print("Orders Table Created")


Orders Table Created


## 3. Create Customers Table

In [ ]:
drop_customers_table = '''
DROP TABLE IF EXISTS customers;
'''

conn.execute(drop_customers_table)

create_customers_table = '''
CREATE TABLE customers (
    customer_id       TEXT PRIMARY KEY,
    customer_name     TEXT,
    segment           TEXT,
    city              TEXT,
    region            TEXT
);
'''

conn.execute(create_customers_table)

print("Customers Table Created")


Customers Table Created


## Upload orders.csv and customers.csv into colab

In [ ]:
import glob
import os

for path in glob.glob("/content/*.csv"):
    os.remove(path)

In [ ]:
from google.colab import files

uploaded_1 = files.upload()

Saving clean_orders.csv to clean_orders.csv


In [ ]:
from google.colab import files

uploaded_2 = files.upload()

Saving customers.csv to customers.csv


In [ ]:
import pandas as pd

orders_df = pd.read_csv("clean_orders.csv") # from Lab 1
customers_df = pd.read_csv("customers.csv")

## Load orders_df into orders table in lab2.db

In [ ]:
import sqlite3

conn = sqlite3.connect("lab2.db")

orders_df.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

customers_df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)


8

##

# Read Orders Table

In [ ]:
query = '''
SELECT *
FROM orders
'''

o_df = pd.read_sql(query, conn)

o_df

,order_id,customer_id,order_date,product_category,quantity,unit_price,delivery_days,status
0,O1001,C001,2026-01-03,Electronics,1,15900.0,3.0,Failed
1,O1002,C002,2026-01-05,Office Supplies,4,120.0,8.0,Failed
2,O1003,C003,2026-01-06,Furniture,1,8900.0,7.0,Succeeded
3,O1004,C001,2026-01-10,Office Supplies,2,350.0,10.0,Succeeded
4,O1005,C004,2026-01-12,Electronics,2,1290.0,5.0,Succeeded
5,O1006,C005,2026-01-14,Furniture,1,4500.0,NaN,Failed
6,O1007,C006,2026-01-18,Office Supplies,10,45.0,2.0,Succeeded
7,O1008,C007,2026-01-21,Electronics,1,25900.0,12.0,Refunded
8,O1009,C008,2026-01-23,Furniture,2,3200.0,6.0,Succeeded
9,O1011,C003,2026-02-01,Unknown,5,85.0,NaN,Refunded


## Read Customers Table

In [ ]:
customer_query = '''
SELECT *
FROM customers
'''

customers_df = pd.read_sql(
    customer_query,
    conn
)

customers_df

,customer_id,customer_name,segment,city,region
0,C001,Anan Trading,Corporate,Bangkok,Central
1,C002,Boonmee Store,Small Business,Chiang Mai,North
2,C003,Chalida Home,Consumer,Khon Kaen,Northeast
3,C004,Dee Digital,Corporate,Bangkok,Central
4,C005,Ekamai Living,Consumer,Chonburi,East
5,C006,Fah Office,Small Business,Nakhon Ratchasima,Northeast
6,C007,Green Market,Consumer,Phuket,South
7,C008,Horizon Solutions,Corporate,Nonthaburi,Central


## JOIN Orders and Customers

In [ ]:
# calculate the total_amount from order's quantity * order's unit_price
join_query = '''
SELECT
    o.order_id,
    c.customer_name,
    c.segment,
    o.quantity * o.unit_price as total_amount,
    o.status
FROM orders o
LEFT JOIN customers c
ON o.customer_id = c.customer_id
'''

joined_df = pd.read_sql(
    join_query,
    conn
)

joined_df

,order_id,customer_name,segment,total_amount,status
0,O1001,Anan Trading,Corporate,15900.0,Failed
1,O1002,Boonmee Store,Small Business,480.0,Failed
2,O1003,Chalida Home,Consumer,8900.0,Succeeded
3,O1004,Anan Trading,Corporate,700.0,Succeeded
4,O1005,Dee Digital,Corporate,2580.0,Succeeded
5,O1006,Ekamai Living,Consumer,4500.0,Failed
6,O1007,Fah Office,Small Business,450.0,Succeeded
7,O1008,Green Market,Consumer,25900.0,Refunded
8,O1009,Horizon Solutions,Corporate,6400.0,Succeeded
9,O1011,Chalida Home,Consumer,425.0,Refunded


## Aggregate Revenue by Segment

In [ ]:
aggregation_query = '''
SELECT
    c.segment,
    SUM(o.quantity * o.unit_price) AS total_revenue
FROM orders o
LEFT JOIN customers c
ON o.customer_id = c.customer_id
GROUP BY c.segment
'''

revenue_df = pd.read_sql(
    aggregation_query,
    conn
)

revenue_df

,segment,total_revenue
0,Consumer,59015.0
1,Corporate,39980.0
2,Small Business,930.0


## What Wrong with the above aggregate revenue??

# Filter Succeeded Status

In [ ]:
delivered_query = '''
SELECT *
FROM orders
WHERE status = 'Succeeded'
'''

delivered_df = pd.read_sql(
    delivered_query,
    conn
)

delivered_df

,order_id,customer_id,order_date,product_category,quantity,unit_price,delivery_days,status
0,O1003,C003,2026-01-06,Furniture,1,8900.0,7.0,Succeeded
1,O1004,C001,2026-01-10,Office Supplies,2,350.0,10.0,Succeeded
2,O1005,C004,2026-01-12,Electronics,2,1290.0,5.0,Succeeded
3,O1007,C006,2026-01-18,Office Supplies,10,45.0,2.0,Succeeded
4,O1009,C008,2026-01-23,Furniture,2,3200.0,6.0,Succeeded
5,O1013,C005,None,Electronics,2,2450.0,7.0,Succeeded
6,O1015,C007,2026-02-15,Furniture,1,11900.0,15.0,Succeeded
7,O1017,C001,2026-02-22,Furniture,2,2800.0,8.0,Succeeded
8,O1018,C003,2026-02-25,Electronics,1,1890.0,6.0,Succeeded
9,O1019,C005,2026-03-02,Office Supplies,8,75.0,3.0,Succeeded


# Revise Aggregate Revenue by Segment

In [ ]:
revised_aggregation_query = '''
SELECT
    c.segment,
    SUM(o.quantity * o.unit_price) AS total_revenue
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.status = 'Succeeded'
GROUP BY c.segment;
'''

revised_revenue_df = pd.read_sql(
    revised_aggregation_query,
    conn
)

revised_revenue_df

,segment,total_revenue
0,Consumer,28190.0
1,Corporate,20880.0
2,Small Business,450.0


## 12. Load Data into Sales Mart

In [ ]:
joined_df.to_sql(
    "sales_mart",
    conn,
    if_exists="replace",
    index=False
)

print("Sales Mart Loaded")


Sales Mart Loaded


## 13. Read Sales Mart

In [ ]:
sales_mart_query = '''
SELECT *
FROM sales_mart
'''

sales_mart_df = pd.read_sql(
    sales_mart_query,
    conn
)

sales_mart_df

,order_id,customer_name,segment,total_amount,status
0,O1001,Anan Trading,Corporate,15900.0,Failed
1,O1002,Boonmee Store,Small Business,480.0,Failed
2,O1003,Chalida Home,Consumer,8900.0,Succeeded
3,O1004,Anan Trading,Corporate,700.0,Succeeded
4,O1005,Dee Digital,Corporate,2580.0,Succeeded
5,O1006,Ekamai Living,Consumer,4500.0,Failed
6,O1007,Fah Office,Small Business,450.0,Succeeded
7,O1008,Green Market,Consumer,25900.0,Refunded
8,O1009,Horizon Solutions,Corporate,6400.0,Succeeded
9,O1011,Chalida Home,Consumer,425.0,Refunded


## 14. Row Count Validation

In [ ]:
row_count_query = '''
SELECT COUNT(*) AS total_rows
FROM sales_mart
'''

row_count_df = pd.read_sql(
    row_count_query,
    conn
)

row_count_df

,total_rows
0,17


## 15. Revenue Reconciliation

In [ ]:
revenue_check_query = '''
SELECT
    SUM(total_amount) AS total_revenue
FROM sales_mart
'''

revenue_check_df = pd.read_sql(
    revenue_check_query,
    conn
)

revenue_check_df

,total_revenue
0,99925.0


# what Wrong with the above reconcilation?

In [ ]:
revised_revenue_check_query = '''
SELECT
    SUM(total_amount) AS total_revenue
FROM sales_mart
WHERE status = 'Succeeded'
'''

revised_revenue_check_df = pd.read_sql(
    revised_revenue_check_query,
    conn
)

revised_revenue_check_df

,total_revenue
0,49520.0


## 16. Create Pipeline Run Log Table

In [ ]:
drop_table_pipeline_run_log = '''
DROP TABLE IF EXISTS pipeline_run_log
'''

conn.execute(drop_table_pipeline_run_log)

create_log_table = '''
CREATE TABLE IF NOT EXISTS pipeline_run_log (
    run_id TEXT,
    workflow_name TEXT,
    row_count INTEGER,
    status TEXT
)
'''

conn.execute(create_log_table)

print("Pipeline Log Table Created")


Pipeline Log Table Created


## 17. Insert Pipeline Run Metadata

In [ ]:
log_data = [
    ("RUN001", "build_sales_mart", 4, "SUCCESS")
]

insert_log_query = '''
INSERT INTO pipeline_run_log
VALUES (?, ?, ?, ?)
'''

conn.executemany(
    insert_log_query,
    log_data
)

conn.commit()

print("Pipeline Metadata Logged")


Pipeline Metadata Logged


## 18. Read Pipeline Run Log

In [ ]:
log_query = '''
SELECT *
FROM pipeline_run_log
'''

log_df = pd.read_sql(
    log_query,
    conn
)

log_df

,run_id,workflow_name,row_count,status
0,RUN001,build_sales_mart,4,SUCCESS


## 19. Close Database Connection

In [ ]:
conn.close()

print("Database Connection Closed")


Database Connection Closed



# Final Learning Outcome

ผู้เรียนควรเข้าใจ:
- SQL + Python integration
- Database connection
- Create tables
- Insert data
- SELECT queries
- JOIN operations
- Aggregation
- Load analytics mart
- Pipeline metadata logging
